# Estimating sales price of a house/apartment

Using a query, you should try to obtain three data files each contains 10000 observations of stars, galaxies, and quasars, respectively, which are distinctly different objects.

As this is data resuting from a (raw) query, it has got several missing/9999 values and other flaws to consider, before applying any further data analysis. You need to fix this!

Thus, this exercise first consists of **Inspecting and Cleaning the data**.

### Features:
The data features / input variables (X) are:
* ID:   unique object ID in the database
* ra:   right ascension (coordinate)
* dec:  declination (coordinate)
* istar:        log-likelihood that the object is point-like, given by the pipeline run on the images
* gmag:  magnitude in g-band
* rmag: magnitude in r-band
* imag: magnitude in i-band
* zmag: magnitude in z-band
* W1:   magnitude in W1-band (from AllWISE)
* W2:   magnitude in W2-band (from AllWISE)
* psfgmag:      PSF magnitude in g-band (i.e. the best-fit magnitude of a point-like object fit to the pixel data)
* psfrmag:      PSF magnitude in r-band (i.e. the best-fit magnitude of a point-like object fit to the pixel data)
* psfimag:      PSF magnitude in i-band (i.e. the best-fit magnitude of a point-like object fit to the pixel data)
* W3:   magnitude in W3-band (from AllWISE)
* W3err:        uncertainty on magnitude in W1-band (from AllWISE)
* J:    magnitude in J-band (from 2MASS, in AllWISE)
* Jerr: uncertainty on J
* H:    magnitude in J-band (from 2MASS, in AllWISE)
* Herr: uncertainty on H
* K:    magnitude in J-band (from 2MASS, in AllWISE)
* Kerr: uncertainty on K
* umag: magnitude in u-band
* zs: "true" redshift

Make sure that you shortly think about (and discuss) which of these features should be included, if you want to try to identify which type of object it is.

Also, this time there is no target value (Y) given in the data. However, given the query selection (by other means) to be stars, galaxies, and quasars, you can consider the file type to be the target. But you need to put these three files together and add a column with the target (i.e. file origin) value.


### Task:
Thus, the task before you is to:<br>
1) Make three queries, which produces three files of data containing stars, galaxies, and quasars.<br>
2) Combine the three data files into one, which has a target value corresponding to the file type.<br>
3) Read and inspect this data, and make sure that you understand what it (roughly) looks like.<br>
4) Clean/cut (or impute) the data, such that different (unsupervised) analysis techniques will work.<br>
5) Run a (k)PCA (and later other techniques) on it, and see what the resulting distributions looks like.<br>

Do you in the end manage to get e.g. get three well separated classes out?<br>

***

* Author: Troels C. Petersen (NBI) & Christian Michelsen (NBI)
* Email:  petersen@nbi.dk
* Date:   4th of May 2026 (latest version)

In [ ]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
from matplotlib import pyplot as plt
import numpy as np
import pandas

import math
from array import array

import numpy as np
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt

# For better looking text on figures
from matplotlib import rc
rc('text', usetex=False)
rc('font', family='serif')

### Class definitions

We define a class for the columns that we are interested in. It holds the index at which it appears, the value for each entry (that the file reading loop assigns), a boolean value that tells us if the value is properly stored in the list, and finally a value that we set in the loop further down which tells us if the value is useable (see below example).

In [ ]:
class variable:
    def __init__( self , column_index ) :
        self.column_index = column_index
        self.value = -1
        self.isValid = False
        self.isGood = False # You set this to True in the loop if the value makes sense
                            # Note that this does not mean that the value is actually good
                            # Example: If condition is -1, the value is valid but bad

In [ ]:
# Whether to save all the plots to disk
SavePlots = False

# Whether to print extra information, debugging goodies, etc.
Verbose = True

# Here you add all the columns from the dataset that you are actually
# interested in.
variable_list = {
# name         : variable( column_index )
  "postalcode" : variable( 4  ) ,
  "salesprice" : variable( 11 ) , # The estimated price? Often 0
  "salesdate"  : variable( 12 ) , # The actual sales price
  "address"    : variable( 25 ) ,
  "sizein"     : variable( 32 ) , # in-door area, whole apartment
  "sizeout"    : variable( 34 ) , # garden, whole lot
  "condition"  : variable( 52 ) , # 1-6
  "pricepaid"  : variable( 89 )
}

###  Variable declarations

We make a counter for each column that stores validness and one that stores validness+goodness. We will use them to consider the quality of whole columns and thereby their usefulness in later analysis.

In [ ]:
valid_list = { name: 0 for name, var in variable_list.items() }
valid_and_good_list = { name: 0 for name, var in variable_list.items() }

# All the rows that made it past the essential checks are saved here
values = { name: [] for name , var in variable_list.items() }

# A counter for each row (even if all the values are invalid).
num_rows = 0

# A counter for all the rows where all the essential columns
# are all good.
num_essential_rows = 0

# Screen resolution (used to convert inches to pixels for the figures)
my_dpi = 96

###  User-defined helper functions

In [ ]:
# "is Not a Number" that works on strings
def isNaN(num) :
    return num != num

# If the entry is valid
def isValid( num ) :
    return not isNaN( num ) and num != "NULL"

# We should always CHECK FIRST if we can convert a value from string
# to a number! Integer conversion has another funny pithole:
# If we try to convert a string "10.0" to integer, python actually
# throws an exception. So we should check first to make sure!
def isInt(s):
    try: 
        int(s)
        return True
    except ValueError:
        return False

# Same deal: Check first before converting!
def isFloat(s):
    try: 
        float(s)
        return True
    except ValueError:
        return False

# Square function
def sqr(a) : return a*a

# Stat.box in figures
def DrawEntryBox( el_list , x_min , x_max , x = 0.85 , y = 0.90 ) :
    mean , rms = MeanRMS( el_list , x_min , x_max )
    
    textstr = r'\begin{eqnarray*}\mathrm{Entries}& %d\\\mu& %.3f\\\sigma& %.3f\end{eqnarray*}'%(len(el_list),mean,rms)
    
    # these are matplotlib.patch.Patch properties
    props = dict(facecolor='white',edgecolor='black', pad=20.0)
    
    # place a text box in upper left in axes coords
    plt.gca().text(x,y, textstr, transform=plt.gca().transAxes, fontsize=14,
            verticalalignment='top', bbox=props)

# Mean and RMS of a list of values ( support ranges )
def MeanRMS( inlist , x_min = None , x_max = None ) :
    if x_min is None or x_max is None :
        mean = np.mean(inlist)
        rms = math.sqrt(np.var(inlist))
    else :
        inlist_array = np.array(inlist)
        inlist_ranged = inlist_array[(inlist_array > x_min) & (inlist_array <x_max)]
        mean = np.mean(inlist_ranged)
        rms = math.sqrt(np.var(inlist_ranged))
    
    return ( mean , rms )

# Mean and RMS of a list of values ( support ranges ) and a list of values used when drawing a Gaussian from the mean and sigma
def MeanRMSXlist( inlist , x_min = None , x_max = None ) :
    if x_min is None or x_max is None :
        mean = np.mean(inlist)
        rms = math.sqrt(np.var(inlist))
        x_fit = np.linspace(min(inlist), max(inlist),100)
    else :
        inlist_array = np.array(inlist)
        inlist_ranged = inlist_array[(inlist_array > x_min) & (inlist_array <x_max)]
        mean = np.mean(inlist_ranged)
        rms = math.sqrt(np.var(inlist_ranged))
        x_fit = np.linspace(min(inlist_ranged), max(inlist_ranged),100)
    
    return ( mean , rms , x_fit )

# Plot linear fit
def PlotLinearFit( x , y , color = "blue" , linewidth = 2.0 ) :    
    plt.plot(np.unique(x), np.poly1d(np.polyfit(x, y, 1))(np.unique(x)),'b-', color=color , linewidth=linewidth)

# Plot linear fit, forced through the origin
def PlotLinearFitThroughOrigin( x , y ) :
    x_ = np.array(x)
    x_ = x_[:,np.newaxis]
    a, _, _, _ = np.linalg.lstsq(x_, y)
    plt.plot(x_, a*x_, 'r-', linewidth=2.0)

***

# Main analysis

## Reading in the data:

This part includes checking the data, and getting basic counting statistics.

In [ ]:
# Open file
f = open("HousingPrices.csv", "r")

# Read, print (if verbose) and ignore header lines
header1 = f.readline()
header2 = header1.strip()
header3 = header2.split(";")
if ( Verbose ) :
    for i in range( len(header3) ) :
        print(i, header3[i] )

# Loop over lines and extract variables of interest
for line in f :
    # Cut away spaces at the ends
    line = line.strip()
    
    # Split the row with ";" as delimeter
    columns = line.split(";")
    
    # Read another row
    num_rows += 1
    
    # Every essential variable increments this number.
    # If the number is exactly X, all essential variables
    # passed.
    essentials_passed = 0
    
    # Loop over every variable in our list of variables
    for name, var in variable_list.items() :
        # Reset previous value
        var.value = -1
        var.isValid = False
        var.isGood = False
        
        # Retrieve the column value from the current row,
        # and remove any spaces that might be at the beginning
        # or end of it
        val = columns[var.column_index].strip()
        
        # If the value is valid move on to test goodness. If
        # it is not valid, do not bother checking further.
        if ( isValid(val) ) :
            var.isValid = True
            
            # If it is the postal code column, we should first check
            # if the entry is an integer before we check the value itself.
            # If the value is not an integer, the typecast will throw an
            # exception that kills our code!
            if ( name == "postalcode" and isInt(val) ) :
                # All good! We save the value as an integer
                var.value = int(val)
                
                # We check if the value is in the postal code range
                # which is probably between 0 and 10000 to a good
                # first approximation.
                var.isGood = ( var.value > 0 and var.value < 10000 )
                
                # If is good, increment essentials passed
                essentials_passed += var.isGood
            
            # Sales price is rounded down to 1/100 of our currency,
            # so it is a float.
            if ( name == "salesprice" and isFloat(val) ) :
                # All good! We save the value as a float
                var.value = float(val)
                
                # Of course the value should be positive.
                # But can we set an upper limit now, or should
                # we reserve that for later? We could set some
                # absurd upper limit like 100 billion. Any number
                # above this limit is quite likely corrupt.
                var.isGood = ( var.value > 0 and var.value < 1e11 )
            
            # The sales date is a string on the form YYYY-MM-DD.
            # Of course we should check to make sure!
            if ( name == "salesdate" ) :
                var.value = val
                # We are lazy, so we will just make sure that the day
                # is not in the future.
                month = int(var.value[5:7])
                year = int(var.value[0:4])
                months_since = 12*(2016-year)+(8-month)
                var.isGood = ( months_since >= 0 )
                # Now, we should also check that DD, MM, and YYYY are
                # valid: Is 1<=MM<=12, does DD exist in month MM, etc.
                # But let us just assume that for now.
                
                essentials_passed += var.isGood
            
            # The adress is just a string. It must not be empty!
            if ( name == "address" ) :
                var.value = val
                var.isGood = bool(var.value)
                essentials_passed += var.isGood
            
            # The in-door size can be 0, if only the whole lot/garden has
            # been considered. BUT we should check that sizein+sizeout is
            # not zero.
            # For this example we require that the indoor area has been
            # stored!
            if ( name == "sizein" and isInt(val) ) :
                var.value = int(val)
                var.isGood = ( var.value > 0 )
                essentials_passed += var.isGood
            
            # The out-door size can be 0, if it is e.g. an apartment
            if ( name == "sizeout" and isInt(val) ) :
                var.value = int(val)
                var.isGood = ( var.value >= 0 )
            
            if ( name == "condition" and isInt(val) ) :
                var.value = int(val)
                var.isGood = ( var.value >= 1 and var.value <= 6 )
                essentials_passed += var.isGood
            
            # This seems to always be an integer, but
            # we shouldn't restrain ourselves yet.
            if ( name == "pricepaid" and isFloat(val) ) :
                var.value = float(val)
                var.isGood = ( var.value > 0 and var.value < 1e11 )
                essentials_passed += var.isGood
        
        # Statistics on quality of our data
        valid_list[name] += var.isValid
        valid_and_good_list[name] += var.isValid and var.isGood
    
    # We have retrieved the values we are interested in.
    # Let's check if we have all the essentials.
    if ( essentials_passed == 6 ) :
        num_essential_rows += 1
    # Else move on to the next row!
    else :
        continue
   
    # We have made it. Save this row!
    for name, var in variable_list.items() :
        values[name].append( var.value )
   
   # As as test: Show entries for Oesterbro:
   # if ( Verbose and variable_list["postalcode"].value == 2100 ) :
   #     print("  Sales Price: %10.1f    Sales Date: %10s      Size: %6.1f / %6.1f    Condition: %2d    Price Paid: %10.1f    Address: %20s, %4d"%( variable_list["salesprice"].value , variable_list["salesdate"].value , variable_list["sizein"].value , variable_list["sizeout"].value , variable_list["condition"].value , variable_list["pricepaid"].value , variable_list["address"].value , variable_list["postalcode"].value ))

# Output statistics. Automatically filled.
print("\n\n  ------------------------------------------------------------------------")
print("      Total entries:  %7d"%(num_rows))
print("  Essentials passed:  %7d"%(num_essential_rows))
print("  ------------------------------------------------------------------------")
print("            Variable    # Valid         # Valid and good   # Valid but bad")
for name, var in variable_list.items():
    print("  %16s :  %7d  (%3d%%) %7d  (%3d%%)   %7s  (%3d%%)"%( name , valid_list[name] , float(valid_list[name])/float(num_rows)*100.0 , valid_and_good_list[name] , float(valid_and_good_list[name])/float(num_rows)*100.0 , str(valid_list[name]-valid_and_good_list[name]) if (valid_list[name]-valid_and_good_list[name])>0 else "-" , float(valid_list[name]-valid_and_good_list[name])/float(num_rows)*100.0 ))
print("  ------------------------------------------------------------------------")


***

# Data analysis

Having read in the data, we now proceed to inspect it to see, what it contains, and if this makes sense. Producing an array of plots involving the key features/variables is usually a **very** good idea!

In [ ]:
# Let us make some easier variables (in order of importance)
pricepaid = values["pricepaid"]
address = values["address"]
postalcode = values["postalcode"]
sizein = values["sizein"]
salesdate = values["salesdate"]
condition = values["condition"]
salesprice = values["salesprice"]
sizeout = values["sizeout"]

# --------------------------------------------------------- #
#  Figure 1(a): Is the info there?
#  Figure 1(b): How much do we lose from our essentials?
# --------------------------------------------------------- #
# Calculate some values
names = [ name for name , var in variable_list.items() ]
effs = [ float(valid_and_good_list[name])/float(num_rows) for name , var in variable_list.items() ]
names_essentials = ( "pricepaid" , "address" , "postalcode" , "sizein" , "salesdate" , "condition" )
vars_in_order_of_importance = ( float(valid_and_good_list["pricepaid"])/float(num_rows) , float(valid_and_good_list["address"])/float(num_rows) , float(valid_and_good_list["postalcode"])/float(num_rows) , float(valid_and_good_list["sizein"])/float(num_rows) , float(valid_and_good_list["salesdate"])/float(num_rows) , float(valid_and_good_list["condition"])/float(num_rows) )
effs_accumulated = [ np.prod(vars_in_order_of_importance[1:i]) for i in range(1,1+len(vars_in_order_of_importance)) ]

# Set up figure
plt.figure(figsize=(800/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figures 1(a) and 1(b)')

plt.subplot( 2 , 1 , 1 )
plt.bar(range(len(names)), effs, align='center')
plt.xticks(range(len(names)), names, size='small')
plt.ylabel('Efficiency')
plt.title('Figure 1(a): Is the info there?')

plt.subplot( 2 , 1 , 2 )
plt.bar(range(len(names_essentials)), effs_accumulated, align='center')
plt.xticks(range(len(names_essentials)), names_essentials, size='small')
plt.ylabel('Accumulated efficiency')
plt.title('Figure 1(b): How much do we lose from our essentials?')

# Draw
plt.tight_layout()
if (SavePlots) : plt.savefig("Figures1ab.pdf", bbox_inches='tight') # Save plot (format follow extension name)
plt.draw()

# --------------------------------------------------------- #
#  Figure 2(a): Is the info correct?
#  Figure 2(b): Any outliers?
# --------------------------------------------------------- #
# Price per area
# We have already checked that the values are valid
# and good, so no worries with division by zero.
price_per_m2 = [ pricepaid[i] / sizein[i] for i in range(0,len(sizein)) ]
log_price_per_m2 = [ math.log10(pricepaid[i] / sizein[i]) for i in range(0,len(sizein)) ]

plt.figure(figsize=(1200/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figures 2(a) and 2(b)')

plt.subplot( 1 , 2 , 1 )
plt.hist( price_per_m2 , 100 , [0.0,5.e4] , histtype = 'step' )
plt.xlabel(r'Price paid per in-door area (DKK/m^{2})')
plt.ylabel('Listings')
plt.title('Figure 2(a): Is the info correct?')
plt.gca().set_yscale("log")

plt.subplot( 1 , 2 , 2 )
plt.hist( log_price_per_m2 , 100 , [0.0,8] , density=True , histtype = 'step' )
plt.xlabel('Log of price paid per in-door area (DKK/m^{2})')
plt.ylabel('Listings (normalized)')
plt.title('Figure 2(b): Any outliers?')
plt.gca().set_yscale("log")

plt.tight_layout()
if (SavePlots) : plt.savefig("Figures2ab.pdf", bbox_inches='tight')
plt.draw()

# --------------------------------------------------------- #
#  Figure 3: price vs m2
# --------------------------------------------------------- #
plt.figure(figsize=(1200/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figure 3: price vs m2')
plt.scatter(pricepaid , sizein)
plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 3: price vs m2')

# Whoops - let us remove the few outliers
#plt.xlim([0,1.3e7])

plt.tight_layout()
if (SavePlots) : plt.savefig("Figure3.pdf", bbox_inches='tight')
plt.draw()

# --------------------------------------------------------- #
#  Figure 4(a): (price vs m2) for condition 1
#  Figure 4(b): (price vs m2) for condition 6
# --------------------------------------------------------- #
# Now we categorise the price vs area scatter plots
# by their condition values for some postal codes.
price_area_per_condition = {
    "all"  : [[[],[]],[[],[]],[[],[]],[[],[]],[[],[]],[[],[]]] ,
    "2100" : [[[],[]],[[],[]],[[],[]],[[],[]],[[],[]],[[],[]]] ,
    "2200" : [[[],[]],[[],[]],[[],[]],[[],[]],[[],[]],[[],[]]] ,
    "2300" : [[[],[]],[[],[]],[[],[]],[[],[]],[[],[]],[[],[]]] ,
    "cities" : [[[],[]],[[],[]],[[],[]],[[],[]],[[],[]],[[],[]]] ,
    "countryside" : [[[],[]],[[],[]],[[],[]],[[],[]],[[],[]],[[],[]]]
}
price_area_postal = {
    "all"  : [[],[],[]] ,
    "2100" : [[],[],[]] ,
    "2200" : [[],[],[]] ,
    "2300" : [[],[],[]] ,
    "cities" : [[],[],[]] ,
    "countryside" : [[],[],[]]
}
for (cond,price,size,postal,date) in zip(condition,pricepaid,sizein,postalcode,salesdate) :
    # Let us cut out all the outliers (more than 5M DKK and larger than 300 m^2)
    if ( price > 5.e6 or size > 300 ) : continue
    month = int(date[5:7])
    year = int(date[0:4])
    months_since = 12*(2016-year)+(8-month)
    
    price_area_per_condition["all"][cond-1][0].append(price)
    price_area_per_condition["all"][cond-1][1].append(size)
    price_area_postal["all"][0].append(price)
    price_area_postal["all"][1].append(size)
    price_area_postal["all"][2].append(months_since)
    # Let us make some categories
    if ( postal == 2100 or postal == 2200 or postal == 2300 ) :
        price_area_per_condition[str(postal)][cond-1][0].append(price)
        price_area_per_condition[str(postal)][cond-1][1].append(size)
        price_area_postal[str(postal)][0].append(price)
        price_area_postal[str(postal)][1].append(size)
        price_area_postal[str(postal)][2].append(months_since)
    if ( postal > 1000 and postal < 3000 ) or ( postal > 8000 and postal < 8200 ) or ( postal > 5000 and postal < 5200 ) :
        price_area_per_condition["cities"][cond-1][0].append(price)
        price_area_per_condition["cities"][cond-1][1].append(size)
        price_area_postal["cities"][0].append(price)
        price_area_postal["cities"][1].append(size)
        price_area_postal["cities"][2].append(months_since)
    else :
        price_area_per_condition["countryside"][cond-1][0].append(price)
        price_area_per_condition["countryside"][cond-1][1].append(size)
        price_area_postal["countryside"][0].append(price)
        price_area_postal["countryside"][1].append(size)
        price_area_postal["countryside"][2].append(months_since)

plt.figure(figsize=(1200/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figures 4(a) and 4(b)')

# Coniditon 2
plt.subplot( 1 , 2 , 1 )

# Get price and area for conditon 2
x = price_area_per_condition["all"][1][0]
y = price_area_per_condition["all"][1][1]

# Plot scatter plot
plt.scatter( x , y , color="green" )

# Plot linear fits
PlotLinearFit( x , y )
PlotLinearFitThroughOrigin( x , y )

plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 4(a): price vs m2 for condition 2')
plt.xlim([0,0.51e7])
plt.ylim([0,300])

# Coniditon 6
plt.subplot( 1 , 2 , 2 )

# Get price and area for conditon 6
x = price_area_per_condition["all"][5][0]
y = price_area_per_condition["all"][5][1]

# Plot scatter plot
plt.scatter( x , y , color="green" )

# Plot linear fit
PlotLinearFit( x , y )

plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 4(b): price vs m2 for condition 6')
plt.xlim([0,0.51e7])
plt.ylim([0,300])

plt.tight_layout()
if (SavePlots) : plt.savefig("Figure4ab.pdf", bbox_inches='tight')
plt.draw()

# --------------------------------------------------------- #
#  Figure 5(a): price vs m2 for each condition
#  Figure 5(b): price vs m2 for each condition, fitted
# --------------------------------------------------------- #
plt.figure(figsize=(1600/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figures 5(a) and 5(b)')

colors = [ "red" , "green" , "blue" , "orange" , "cyan" , "purple" ]
scatter_handles = []

# All conditions in each their color
plt.subplot( 1 , 2 , 1 )
for i in reversed(range(0,6)) :
    x = price_area_per_condition["all"][i][0]
    y = price_area_per_condition["all"][i][1]
    scatter_handles.append(plt.scatter( x , y , color=colors[i], label="Condition %d"%(i+1)))

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 5(a): price vs m2 for each condition')
plt.xlim([0,0.55e7])
plt.ylim([0,350])

# All conditions in one color with fits in each their color
plt.subplot( 1 , 2 , 2 )
for i in reversed(range(0,6)) :
    x = price_area_per_condition["all"][i][0]
    y = price_area_per_condition["all"][i][1]
    plt.scatter( x , y , color="black", label="Condition %d"%(i+1))
    PlotLinearFit( x , y , colors[i] , 3 )

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 5(b): price vs m2 for each condition, fitted')
plt.xlim([0,0.55e7])
plt.ylim([0,350])

plt.tight_layout()
if (SavePlots) : plt.savefig("Figures5ab.pdf", bbox_inches='tight')
plt.draw()

# --------------------------------------------------------- #
#  Figure 6(a): price vs m2 for each condition for postal 2100
#  Figure 6(b): price vs m2 for each condition for postal 2100, fitted
# --------------------------------------------------------- #
plt.figure(figsize=(1600/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figures 6(a) and 6(b)')

scatter_handles = []

# All conditions in each their color
plt.subplot( 1 , 2 , 1 )
for i in reversed(range(0,6)) :
    x = price_area_per_condition["2100"][i][0]
    y = price_area_per_condition["2100"][i][1]
    scatter_handles.append(plt.scatter( x , y , color=colors[i], label="Condition %d"%(i+1)))

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 6(a): price vs m2 for each condition for postal 2100')
plt.xlim([0,0.55e7])
plt.ylim([0,350])

# All conditions in one color with fits in each their color
plt.subplot( 1 , 2 , 2 )
for i in reversed(range(0,6)) :
    x = price_area_per_condition["2100"][i][0]
    y = price_area_per_condition["2100"][i][1]
    plt.scatter( x , y , color="black", label="Condition %d"%(i+1))
    PlotLinearFit( x , y , colors[i] , 3 )

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 6(b): price vs m2 for each condition for postal 2100, fitted')
plt.xlim([0,0.55e7])
plt.ylim([0,350])

plt.tight_layout()
if (SavePlots) : plt.savefig("Figures6ab.pdf", bbox_inches='tight')
plt.draw()

# --------------------------------------------------------- #
#  Figure 7(a): price vs m2 for some postal codes
#  Figure 7(b): Histogram of prices for some postal codes
#  Figure 7(c): Histogram of sizes for some postal codes
#  Figure 7(d): Histogram of price/size for some postal codes
# --------------------------------------------------------- #
plt.figure(figsize=(1600/my_dpi, 800/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figures 7(a), 7(b), 7(c), and 7(d)')

colors_post = {"2100": "red" , "2200": "green", "2300": "blue"}
scatter_handles = []

# Price vs m2 for three postal codes
plt.subplot( 2 , 2 , 1 )
for post in reversed(["2100", "2200", "2300"]) :
    x = price_area_postal[post][0]
    y = price_area_postal[post][1]
    scatter_handles.append(plt.scatter( x , y , color=colors_post[post], label="Postalcode %s"%(post)))

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid (DKK)')
plt.ylabel('In-door area (m^{2})')
plt.title('Figure 7(a): price vs m2 for some postal codes')
plt.xlim([0,0.55e7])
plt.ylim([0,350])

# Histogram of their prices
plt.subplot( 2 , 2 , 2 )
for post in reversed(["2100", "2200", "2300"]) :
    x = price_area_postal[post][0]
    plt.hist( x , 50 , [0,0.55e7] , histtype = 'step' )

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid (DKK)')
plt.ylabel('Listings')
plt.title('Figure 7(b): Histogram of prices for some postal codes')
plt.xlim([0,0.55e7])

# Histogram of their sizes
plt.subplot( 2 , 2 , 3 )
for post in reversed(["2100", "2200", "2300"]) :
    x = price_area_postal[post][1]
    plt.hist( x , 50 , [0,300] , histtype = 'step' )

plt.legend(handles=scatter_handles)
plt.xlabel('In-door area (m^{2})')
plt.ylabel('Listings')
plt.title('Figure 7(c): Histogram of sizes for some postal codes')

# Histogram of their price/size
plt.subplot( 2 , 2 , 4 )
for post in reversed(["2100", "2200", "2300"]) :
    x = [a/b for a,b in zip(price_area_postal[post][0],price_area_postal[post][1])]
    plt.hist( x , 50 , [0,50e3] , histtype = 'step' )

plt.legend(handles=scatter_handles)
plt.xlabel('Price paid per in-door area (DKK/m^{2})')
plt.ylabel('Listings')
plt.title('Figure 7(d): Histogram of price/size for some postal codes')

plt.tight_layout()
if (SavePlots) : plt.savefig("Figures7abcd.pdf", bbox_inches='tight')
plt.draw()

# --------------------------------------------------------- #
#  Figure 8: Average price development for cities vs countryside
# --------------------------------------------------------- #

country_avg_price = [ 0 for x in range(0,290)]
country_rms_price = [ 0 for x in range(0,290)]
country_num_entries = [ 0 for x in range(0,290)]

city_avg_price = [ 0 for x in range(0,290)]
city_rms_price = [ 0 for x in range(0,290)]
city_num_entries = [ 0 for x in range(0,290)]

for i in range(0,len(price_area_postal["countryside"][0])) :
    months = price_area_postal["countryside"][2][i]
    if ( months > 289 ) : continue
    price = price_area_postal["countryside"][0][i]
    country_avg_price[months] += price
    country_rms_price[months] += price**2
    country_num_entries[months] += 1

for i in range(0,len(price_area_postal["cities"][0])) :
    months = price_area_postal["cities"][2][i]
    if ( months > 289 ) : continue
    price = price_area_postal["cities"][0][i]
    city_avg_price[months] += price
    city_rms_price[months] += price**2
    city_num_entries[months] += 1

for i in range(0,290) :
    if ( country_num_entries[i] > 0 ) :
        country_avg_price[i] = country_avg_price[i] / country_num_entries[i]
        country_rms_price[i] = math.sqrt( country_rms_price[i] / country_num_entries[i] - country_avg_price[i]**2) / math.sqrt(city_num_entries[i])
    else :
        country_avg_price[i] = 0
        country_rms_price[i] = 0
    
    if ( city_num_entries[i] > 0 ) :
        city_avg_price[i] = city_avg_price[i] / city_num_entries[i]
        city_rms_price[i] = math.sqrt( city_rms_price[i] / city_num_entries[i] - city_avg_price[i]**2) / math.sqrt(city_num_entries[i])
    else :
        city_avg_price[i] = 0
        city_rms_price[i] = 0

plt.figure(figsize=(1200/my_dpi, 600/my_dpi), dpi=my_dpi)
plt.gcf().suptitle('Figure 8: Price development for cities vs countryside')

scatter_handles = []

scatter_handles.append(plt.errorbar( range(0,290) , country_avg_price , yerr=country_rms_price, fmt='o' , color="blue", label="Countryside"))
scatter_handles.append(plt.errorbar( range(0,290) , city_avg_price , yerr=city_rms_price, fmt='o' , color="red", label="Cities"))

plt.legend(handles=scatter_handles)
plt.xlabel('Months since today')
plt.ylabel('Price (DKK)')
plt.title('Figure 8: Price development for cities vs countryside')
#plt.xlim([0,0.55e7])
plt.xlim([0,290])

plt.tight_layout()
if (SavePlots) : plt.savefig("Figures8.pdf", bbox_inches='tight')
plt.draw()